In [81]:
from pypdf import PdfReader, PdfWriter
import pandas as pd
import fitz
import re
import spacy
import os 

In [82]:
reader =PdfReader('resumes/Resume - Business Analyst.pdf')
print(len(reader.pages))

2


In [83]:
pageObj = reader.pages[0]
pageObj2 = reader.pages[1]


In [ ]:
print(pageObj.extract_text())

In [85]:
print(pageObj2.extract_text())

●  Authored  comprehensive  training  materials  for  team  members  transitioning  to  new  claims  software    EDUCATION   05/2025  –  12/2025   Code  You/Code  Louisville  Certificate  of  completion      Louisville   2015  –  2017   Indiana  University  Southeast  Bachelor  of  Science,  Accounting     New  Albany   01/2007  –  12/2009   Jefferson  Community  and  Technical  College  Associate  of  Arts     Louisville    CERTIFICATIONS
 OpenClaw  Masterclass:  Install,  Build  &  Deploy  Real  AI  Agents  |  Udemy  |  2026  Vibe  Coding  Bootcamp:  Build  Any  App,  Game  or  Website  with  AI
     SKILLS   
Python  Mac  OS  
Windows  SQL  
Java  Data  Entry  
Data  Validation  Great  Plains  
JD  Edwards  ViewPoint  
CCH  Engagement  Excel  (Pivot  Tables  ,  V-Lookups)  
Process  Documentation  Cross-functional  Communication  
Issue  Resolution  Strong  Research  Skills  
SOP  Documentation     


In [ ]:
# Extract text and iterate pages in pdf

def extract_text_from_pdf(file_path): 
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = extract_text_from_pdf("resumes/Resume - Business Analyst.pdf")
print(resume_text[:500])

In [ ]:
# Process and clean resume

def clean_text(text):
    text = re.sub(r'\n+','\n', text)
    text = re.sub(r' +',' ', text)
    return text.strip()

cleaned  = clean_text(resume_text)
print(cleaned)

In [ ]:
# Extract name, email, phone

def extract_email(text):
    match = re.search(r'\S+@\S+', text)

    if match:
        return match.group(0)
    else:
        None

def extract_phone(text):
    match = re.search(r'\(\d{3}\)\s\d{3}-\d{4}', text)

    if match:
            return match.group(0)
    else:
         None 
         
email = extract_email(cleaned)
phone = extract_phone(cleaned)
print("Email: ", email)#,  "Phone: ", phone)

# Removing phone for Github


Email:  jmasters0013@me.com


In [89]:
# Load english language model 

#nlp = spacy.load('en_core_web_sm')

#def extract_name(text):
    #look at first 100 characters
 #   doc = nlp(text[:100])
  #  for ent in doc.ents:
   #     if ent.label_ == "PERSON":
    #        return ent.text
   # return None

#name = extract_name(resume_text)
#print(name)

# If above does not work, try this

def extract_name(text):
    first_line = text.strip().split('\n')[0]
    return first_line.strip()

name = extract_name(resume_text)
print(name)

Justin Masters


In [90]:
# Custom still Matching from Resume and return matches

SKILL_SET = ['Python', 'SQL', "Excel", 'Power BI', 'Machine Learning', 'Data Analysis']

def extract_skills(text, skills=SKILL_SET):
    found = [skill for skill in skills if skill.lower() in text.lower()]
    return list(set(found))

skills = extract_skills(resume_text)
print(skills)

['Excel', 'Python', 'SQL']


In [91]:
# Extracting Education and Degrees. 

EDU_KEYWORDS = ['Bachelor', 'Graduate Degree', 'B.Sc', 'M.Sc', 'PhD', 'B.E', 'M.E', 'Associate', 'Universoty', 'College', 'Institute', 'Certificate']
# Changed Masters in EDU_KEYWORDS to Garduate Degree. Was returning my name

def extract_education(text):
    lines = text.split('\n')
    education = []

    # Disgined for my resume with words to skip that were causing not to work properly
    skip_words = ['summary', 'experience', 'skills', 'coordinated', 'reconciled', 'prepared']
    for line in lines:
        if any(skip.lower() in line.lower() for skip in skip_words):
            continue
        for word in EDU_KEYWORDS:
            if word.lower() in line.lower() and len(line) > 10:
                education.append(line.strip())
                break
    return education
    
edu = extract_education(resume_text)
print(edu)

['05/2025 – 12/2025 \u200bCode You/Code Louisville Certificate of completion\u200b', 'Indiana University Southeast Bachelor of Science, Accounting\u200b', '01/2007 – 12/2009 \u200bJefferson Community and Technical College Associate of Arts\u200b']


In [ ]:
# Extracting work experience snips

def extract_experience(text):
    text_clean = re.sub(r'[\t]+', ' ', text)
    
    pattern = r'(?:experience|work history|employment)[\s:]*(.*?)(?:education|skills|certifications|$)'
    match = re.search(pattern, text_clean, re.IGNORECASE | re.DOTALL)
    
    if match:
        experience_text = match.group(1).strip()
        return experience_text
    return "Experience section not found" 

raw_experience_block = extract_experience(resume_text)
print(raw_experience_block)
        

in claims analysis, 
data validation, and process documentation with a primary focus on Medicare claims. Proven ability to identify discrepancies, analyze trends, 
and communicate complex findings clearly to cross-functional stakeholders at all levels. Experienced authoring training materials, 
documenting workflows, and coordinating process improvements in compliance-driven healthcare settings. Strong Excel foundation with 
actively developing SQL and Python


In [ ]:
# Final output Dict

parsed_resume = {
    'Name': extract_name(cleaned),
    'Email': extract_email(cleaned),
    'Phone':  extract_phone(cleaned),
    'Skills': extract_skills(cleaned),
    'Education': extract_education(cleaned),
    'Experience': extract_experience(cleaned)
}
 
print(parsed_resume)

In [94]:
# Automate resume folder parsing

def process_folder(folder_path):
    results = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            path = os.path.join(folder_path, file)
            text = clean_text(extract_text_from_pdf(path))
            parsed = {
                'File': file,
                'Name': extract_name(text),
                'Email': extract_email(text),
                'Phone': extract_phone(text),
                'Skills': extract_skills(text),
                'Educaction': extract_education(text),
                'Experience': extract_experience(text)
            }
            results.append(parsed)
    return pd.DataFrame(results)
    
df = process_folder("resumes")
df.to_csv("parsed_resume.csv", index=False)

In [ ]:
print(f"Total Resumes Parsed: {len(df)}")
df.head()